<a href="https://colab.research.google.com/github/aidev-ahmedamr/real-time-dynamic-pricing-engine/blob/main/notebooks/07_mlflow_experiment_tracking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install mlflow xgboost

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132

In [ ]:
import pandas as pd
import numpy as np
import joblib
import mlflow
import mlflow.xgboost

from xgboost import XGBRegressor

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving dynamic_pricing_processed.csv to dynamic_pricing_processed.csv


In [ ]:
df = pd.read_csv(
    "dynamic_pricing_processed.csv"
)

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (72900, 26)


,timestamp,product_id,category,base_price,cost_price,current_price,competitor_price,inventory_level,views_last_hour,add_to_cart_count,...,conversion_rate,demand,price_ratio_to_competitor,price_difference,profit_margin,inventory_ratio,cart_to_view_ratio,avg_demand_3,avg_demand_7,demand_lag_1
0,2025-01-01 00:00:00,P00001,Electronics,457.36,269.82,344.82,521.74,83,44,10,...,0.3182,14,0.660904,-176.92,0.217505,0.932584,0.227273,14.000000,14.000000,9.0
1,2025-01-01 12:00:00,P00001,Electronics,457.36,269.82,542.60,516.45,5,54,12,...,0.0370,2,1.050634,26.15,0.502728,0.056180,0.222222,8.000000,8.000000,14.0
2,2025-01-02 00:00:00,P00001,Electronics,457.36,269.82,378.48,474.69,8,44,7,...,0.0682,3,0.797320,-96.21,0.287096,0.089888,0.159091,6.333333,6.333333,2.0
3,2025-01-02 12:00:00,P00001,Electronics,457.36,269.82,499.18,470.87,19,36,6,...,0.0833,3,1.060123,28.31,0.459474,0.213483,0.166667,2.666667,5.500000,3.0
4,2025-01-03 00:00:00,P00001,Electronics,457.36,269.82,537.41,501.91,44,56,13,...,0.0536,3,1.070730,35.50,0.497925,0.494382,0.232143,3.000000,5.000000,3.0


In [ ]:
uploaded = files.upload()

features = joblib.load(
    "model_features.pkl"
)

print("Number of features:", len(features))
print(features)

Saving model_features.pkl to model_features (1).pkl
Number of features: 22
['base_price', 'cost_price', 'current_price', 'competitor_price', 'inventory_level', 'views_last_hour', 'add_to_cart_count', 'hour', 'day_of_week', 'month', 'is_weekend', 'rating', 'num_reviews', 'conversion_rate', 'price_ratio_to_competitor', 'price_difference', 'profit_margin', 'inventory_ratio', 'cart_to_view_ratio', 'avg_demand_3', 'avg_demand_7', 'demand_lag_1']


In [ ]:
target = "demand"

X = df[features].copy()

y = df[target].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (72900, 22)
y shape: (72900,)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (58320, 22)
X_test: (14580, 22)
y_train: (58320,)
y_test: (14580,)


In [ ]:
mlflow.set_experiment(
    "dynamic-pricing-demand-model"
)

<Experiment: artifact_location='/content/mlruns/1', creation_time=1787740383171, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1787740383171, lifecycle_stage='active', name='dynamic-pricing-demand-model', tags={}, trace_location=None, workspace='default'>

In [ ]:
with mlflow.start_run():

    model = XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=8,
        random_state=42
    )

    model.fit(
        X_train,
        y_train
    )

    predictions = model.predict(
        X_test
    )

    mae = mean_absolute_error(
        y_test,
        predictions
    )

    rmse = mean_squared_error(
        y_test,
        predictions
    ) ** 0.5

    r2 = r2_score(
        y_test,
        predictions
    )

    mlflow.log_params({
        "n_estimators": 300,
        "learning_rate": 0.05,
        "max_depth": 8,
        "random_state": 42
    })

    mlflow.log_metrics({
        "mae": mae,
        "rmse": rmse,
        "r2": r2
    })

    mlflow.xgboost.log_model(
        model,
        "demand_model"
    )

    print("================================")
    print("MLflow Training Completed")
    print("================================")
    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R²   : {r2:.4f}")

2026/08/26 10:39:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


MLflow Training Completed
MAE  : 0.0207
RMSE : 0.1492
R²   : 0.9994


In [ ]:
joblib.dump(
    model,
    "demand_model.pkl"
)

joblib.dump(
    features,
    "model_features.pkl"
)

print("Model saved successfully!")

Model saved successfully!


In [ ]:
import os

print(
    "demand_model.pkl:",
    round(
        os.path.getsize("demand_model.pkl") / (1024 * 1024),
        2
    ),
    "MB"
)

print(
    "model_features.pkl:",
    round(
        os.path.getsize("model_features.pkl") / (1024 * 1024),
        2
    ),
    "MB"
)

demand_model.pkl: 3.23 MB
model_features.pkl: 0.0 MB
